# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
#!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [20]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name= "multi-query-tools", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [8]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

In [11]:
# Data exploration 

import json
import statistics

print(f"Docs: {len(data)}\n")

# 1. ESTRUCTURE
print("Slot available")
for key, val in data[0].items():
    tipo = type(val).__name__
    ejemplo = str(val)[:60]
    print(f"  {key:20s} | {tipo:10s} | ej: {ejemplo}")


Docs: 15

Slot available
  content              | str        | ej: Effective: March 2020
Purpose

The purpose of this full-time
  summary              | str        | ej: This policy outlines the guidelines for full-time remote wor
  name                 | str        | ej: Work From Home Policy
  url                  | str        | ej: ./sharepoint/Work from home policy.txt
  created_on           | str        | ej: 2020-03-01
  updated_at           | str        | ej: 2020-03-01
  category             | str        | ej: teams
  _run_ml_inference    | bool       | ej: True
  rolePermissions      | list       | ej: ['demo', 'manager']


In [12]:
print("Categories distribution")
cats = {}
for doc in data:
    c = doc.get("category", "unknown")
    cats[c] = cats.get(c, 0) + 1
for c, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {c:15s}: {n} documents")


Categories distribution
  sharepoint     : 9 documents
  teams          : 4 documents
  github         : 2 documents


In [13]:
print("LEN(caracteres)")
longitudes = [len(doc["content"]) for doc in data]
print(f"  Min     : {min(longitudes)}")
print(f"  Max     : {max(longitudes)}")
print(f"  Promedio: {statistics.mean(longitudes):.0f}")
print(f"  Mediana : {statistics.median(longitudes):.0f}")
docs_cortos = sum(1 for l in longitudes if l < 800)
print(f"  Docs <800 chars: {docs_cortos}")
print()


LEN(caracteres)
  Min     : 268
  Max     : 3993
  Promedio: 2811
  Mediana : 3173
  Docs <800 chars: 2



In [17]:
print("First 800 characters")
print(data[0]["content"][:800])
print("...")

First 800 characters
Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity and productivity of business operations during the COVID-19 pandemic and beyond.
Scope

This policy applies to all employees who are eligible for remote work as determined by their role and responsibilities. It is designed to allow employees to work from home full time while maintaining the same level of performance and collaboration as they would in the office.
Eligibility

Employees who can perform their work duties remotely and have received approval from their direct supervisor and the HR department are eligible for this work-from-home arrangement.
Equipment and Resources

The necessary equipment an
...


### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [18]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")
    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [21]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="multi-query-tools",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [23]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services to customers in the United States, Canada, Mexico, Central and South America. They work closely with other departments, such as marketing, product development, and customer support, to ensure the delivery of high-quality products and services to clients.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [24]:
ans1 = chain.invoke("what is the most important project that the documents handle") 

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the primary focus of the documents in the database?', '2. Can you suggest the top project that the documents cover?', '3. Which project holds the most significance in the documents?']


In [25]:
print(ans1)


The documents do not specify a specific project as the most important. However, they do outline various projects and initiatives, such as increasing revenue, expanding market share, launching new products, and improving customer satisfaction, that are all important for the company's success.


In [26]:
ans2 = chain.invoke("which new products are the team mates exploring?")

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the latest products being researched by the team members?', '2. Can you suggest any new products that the team is currently exploring?', '3. Which products are the team members currently investigating?']


In [27]:
print(ans2) 



The team is exploring the launch of at least two new products or services in high-demand market segments as part of their objectives for fiscal year 2024. Additionally, they are also continuously evaluating and recommending new technologies, tools, and practices to improve software quality and efficiency.


## Iteration 1

In [28]:
LLM_CONTEXT_PROMPT1 = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. Answer in Spanish and using bulled points of less than 20 words. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT1 = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{category}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT1, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT1 | llm

ans = chain.invoke("what is the nasa projects of sales team?")
print()
print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ["1. What are the current projects being undertaken by NASA's sales team?", "2. Can you provide information on the sales team's projects at NASA?", "3. How does the sales team at NASA contribute to the organization's projects?"]



---- Answer ----

I don't know.


In [29]:

LLM_CONTEXT_PROMPT2 = ChatPromptTemplate.from_template(
    """You are an assistant to Laura Martinez, Area Vice-President of North     America.
    You have access to internal company documents. Use them to answer questions
    from Laura's team members. Be precise and cite which document the info comes from.
    If the documents don't have the answer, say you'd need to check with Laura directly.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT2 = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{category}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT2, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)
chain = _context | LLM_CONTEXT_PROMPT2 | llm

ans4 = chain.invoke("What's the organizational structure for the Americas and how does it relate to engineering?")
print()
print("---- Answer ----")
print(ans4)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the organizational layout for the Americas and its connection to engineering?', '2. Can you explain the organizational structure of the Americas and its relationship with engineering?', '3. How is engineering related to the organizational structure of the Americas?']



---- Answer ----

The Americas region is divided into two main areas: North America and South America. The North America South America region (NASA) has two Area Vice-Presidents: Laura Martinez is the Area Vice-President of North America, and Gary Johnson is the Area Vice-President of South America. This structure is important for engineering as it allows for a dedicated engineering team to support each area and collaborate with the sales teams in those regions to develop and deliver software solutions that meet the unique needs of customers in each market. Additionally, the engineering teams in each region work closely with other departments, such as product development and customer support, to ensure high-quality products and services are consistently delivered to clients.


In [30]:
#Plain Retriever vs MultiQuery Retriever

from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate
from langchain.schema import format_document

# 1. Plain retriever — single query, no variations
plain_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. MultiQuery retriever — generates 3 variations
mq_retriever = MultiQueryRetriever.from_llm(
    vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm,
)

# 3. Same prompt for both
QA_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant to Laura Martinez, Area Vice-President of North America.
    You answer questions from her team using internal company documents.
    Be precise and cite the source document name.

    context: {context}
    Question: "{question}"
    Answer:"""
)

DOC_PROMPT = PromptTemplate.from_template(
    """--- {name} ---
About: {summary}
{page_content}"""
)

def combine(docs):
    return "\n\n".join(format_document(d, DOC_PROMPT) for d in docs)

# 4. Build both chains
def build_chain(retriever):
    _context = RunnableParallel(
        context=retriever | combine,
        question=RunnablePassthrough(),
    )
    return _context | QA_PROMPT | llm

plain_chain = build_chain(plain_retriever)
mq_chain = build_chain(mq_retriever)

# 5. Ask the same question
question = "what is the nasa sales team?"

print("=== PLAIN RETRIEVER (1 query) ===")
plain_docs = plain_retriever.invoke(question)
print(f"Documents found: {len(plain_docs)}")
for d in plain_docs:
    print(f"  • {d.metadata.get('name', 'unknown')}")

print()

print("=== MULTIQUERY RETRIEVER (3 variations) ===")
mq_docs = mq_retriever.invoke(question)
print(f"Documents found: {len(mq_docs)}")
for d in mq_docs:
    print(f"  • {d.metadata.get('name', 'unknown')}")

print()

# 6. Extra docs that MultiQuery found
plain_names = {d.metadata.get('name') for d in plain_docs}
mq_names = {d.metadata.get('name') for d in mq_docs}
extra = mq_names - plain_names
print(f"=== EXTRA DOCS from MultiQuery ===")
if extra:
    for name in extra:
        print(f"  + {name}")
else:
    print("  Same documents — try a different question")
print(f"\nUnique coverage: Plain={len(plain_names)}, MultiQuery={len(mq_names)}")

print("\n--- Answers ---")
print("PLAIN:", plain_chain.invoke(question))
print()
print("MULTIQUERY:", mq_chain.invoke(question))

=== PLAIN RETRIEVER (1 query) ===
Documents found: 3
  • Sales Organization Overview
  • Sales Engineering Collaboration
  • Fy2024 Company Sales Strategy

=== MULTIQUERY RETRIEVER (3 variations) ===


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


Documents found: 3
  • Sales Organization Overview
  • Sales Engineering Collaboration
  • Fy2024 Company Sales Strategy

=== EXTRA DOCS from MultiQuery ===
  Same documents — try a different question

Unique coverage: Plain=3, MultiQuery=3

--- Answers ---
PLAIN:  The NASA sales team is a part of the North America South America region within the company's sales organization. It is led by two Area Vice-Presidents, Laura Martinez and Gary Johnson, and is responsible for promoting and selling the company's products and services in the United States, Canada, Mexico, Central and South America. This information can be found in the "Sales Organization Overview" document.



INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


MULTIQUERY:  The NASA sales team is a part of the North America South America region within the company's sales organization. It is led by two Area Vice-Presidents, Laura Martinez and Gary Johnson, and is responsible for promoting and selling the company's products and services in the United States, Canada, Mexico, Central and South America. This information can be found in the "Sales Organization Overview" document.
